# LLM Benchmark on Google Colab T4 GPU

This notebook demonstrates how to:
1. Load a pre-trained LLM model
2. Run inference benchmarks on Google Colab's T4 GPU
3. Measure performance metrics (latency, throughput, memory usage)
4. Visualize the results

The model used is **DistilBERT** - a lightweight version of BERT that's perfect for testing on limited GPU resources like T4.

## 1. Install Required Libraries

In [ ]:
# Install required libraries
import subprocess
import sys

# Install transformers and torch
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])

print("✓ All libraries installed successfully!")

## 2. Check GPU and Setup

In [ ]:
import torch
import numpy as np
import time
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime

# Check GPU availability
print("GPU Information:")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  Current GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print()

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {device}")

## 3. Load Pretrained LLM Model

We use **DistilBERT** - a lightweight BERT model that works great on T4 GPUs. It's 40% smaller and 60% faster than BERT while retaining 97% of its performance.

In [ ]:
# Load DistilBERT tokenizer and model
print("Loading DistilBERT model...")
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Move model to GPU
model = model.to(device)
model.eval()  # Set to evaluation mode

print(f"✓ Model loaded successfully!")
print(f"  Model: {model_name}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

# Get model size
param_size = sum(p.numel() for p in model.parameters()) * 4 / (1024 ** 2)  # 4 bytes per float32
print(f"  Model size: {param_size:.2f} MB")

## 4. Prepare Benchmark Dataset

Create a simple benchmark dataset with sample texts for inference testing.

In [ ]:
# Create benchmark dataset with sample texts
sample_texts = [
    "This movie is absolutely wonderful and I really enjoyed watching it.",
    "The weather today is quite pleasant, perfect for outdoor activities.",
    "I'm not satisfied with the quality of this product at all.",
    "The new restaurant in town serves excellent food and great service.",
    "This book has an interesting plot and keeps you engaged throughout.",
    "The traffic was terrible today, it took me hours to get home.",
    "I love spending time with my family and friends on weekends.",
    "The conference was informative with many valuable insights shared.",
    "This software is user-friendly and easy to navigate.",
    "The customer service team was extremely helpful and responsive.",
    "The workout today was intense but very rewarding.",
    "I'm impressed with the innovative features of this gadget.",
    "The meeting lasted longer than expected but was productive.",
    "The travel experience was amazing and memorable.",
    "The project deadline is coming up next week.",
] * 4  # Repeat to have 60 samples

print(f"Created benchmark dataset with {len(sample_texts)} samples")
print(f"Sample texts (first 3):")
for i, text in enumerate(sample_texts[:3]):
    print(f"  {i+1}. {text[:60]}...")

# Tokenize the inputs
print("\nTokenizing inputs...")
encodings = tokenizer(
    sample_texts,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

# Move tokens to device
input_ids = encodings["input_ids"].to(device)
attention_mask = encodings["attention_mask"].to(device)

print(f"✓ Tokenization complete!")
print(f"  Input shape: {input_ids.shape}")
print(f"  Max sequence length: 128 tokens")

## 5. Run Inference Benchmark

Execute inference on the benchmark dataset and measure performance metrics.

In [ ]:
# Run inference benchmarks with different batch sizes
batch_sizes = [1, 4, 8, 16, 32]
benchmark_results = []

print("Running inference benchmarks...\n")

for batch_size in batch_sizes:
    print(f"Testing batch size: {batch_size}")
    
    latencies = []
    throughputs = []
    
    # Warm-up run
    with torch.no_grad():
        _ = model(input_ids[:batch_size], attention_mask=attention_mask[:batch_size])
    
    # Benchmark runs
    num_runs = 10
    for _ in range(num_runs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        
        start_time = time.time()
        
        with torch.no_grad():
            outputs = model(
                input_ids[:batch_size],
                attention_mask=attention_mask[:batch_size]
            )
        
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        
        end_time = time.time()
        
        latency = (end_time - start_time) * 1000  # Convert to ms
        throughput = batch_size / (end_time - start_time)  # samples per second
        
        latencies.append(latency)
        throughputs.append(throughput)
    
    # Calculate statistics
    avg_latency = np.mean(latencies)
    std_latency = np.std(latencies)
    avg_throughput = np.mean(throughputs)
    
    benchmark_results.append({
        "Batch Size": batch_size,
        "Avg Latency (ms)": avg_latency,
        "Std Latency (ms)": std_latency,
        "Throughput (samples/s)": avg_throughput
    })
    
    print(f"  Avg Latency: {avg_latency:.2f} ms ± {std_latency:.2f} ms")
    print(f"  Throughput: {avg_throughput:.2f} samples/s\n")

print("✓ Benchmark complete!")

## 6. Measure Performance Metrics

Calculate key metrics including GPU memory usage, model efficiency, and tokens per second.

In [ ]:
# Measure memory usage
print("Memory and Performance Metrics:")
print("=" * 50)

if torch.cuda.is_available():
    peak_memory = torch.cuda.max_memory_allocated() / 1024 ** 2  # Convert to MB
    current_memory = torch.cuda.memory_allocated() / 1024 ** 2
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024 ** 2
    
    print(f"\nGPU Memory Usage:")
    print(f"  Peak Memory: {peak_memory:.2f} MB")
    print(f"  Current Memory: {current_memory:.2f} MB")
    print(f"  Total GPU Memory: {total_memory:.2f} MB")
    print(f"  Memory Utilization: {(peak_memory / total_memory) * 100:.2f}%")

# Calculate tokens per second (multiply throughput by sequence length)
seq_length = 128
best_result = benchmark_results[-1]  # Largest batch size
tokens_per_second = best_result["Throughput (samples/s)"] * seq_length

print(f"\nEfficiency Metrics (Batch Size {best_result['Batch Size']}):")
print(f"  Samples per Second: {best_result['Throughput (samples/s)']:.2f}")
print(f"  Tokens per Second: {tokens_per_second:.2f}")
print(f"  Latency: {best_result['Avg Latency (ms)']:.2f} ms ± {best_result['Std Latency (ms)']:.2f} ms")

# Create results dataframe
df_results = pd.DataFrame(benchmark_results)
print("\nDetailed Benchmark Results:")
print(df_results.to_string(index=False))

## 7. Visualize Results

Create plots to display benchmark results for easy interpretation.

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("DistilBERT Inference Benchmark Results on T4 GPU", fontsize=16, fontweight="bold")

# Plot 1: Latency vs Batch Size
ax1 = axes[0, 0]
ax1.errorbar(
    df_results["Batch Size"],
    df_results["Avg Latency (ms)"],
    yerr=df_results["Std Latency (ms)"],
    marker="o",
    linestyle="-",
    linewidth=2,
    markersize=8,
    capsize=5,
    color="steelblue"
)
ax1.set_xlabel("Batch Size", fontsize=11, fontweight="bold")
ax1.set_ylabel("Latency (ms)", fontsize=11, fontweight="bold")
ax1.set_title("Inference Latency vs Batch Size", fontsize=12, fontweight="bold")
ax1.grid(True, alpha=0.3)

# Plot 2: Throughput vs Batch Size
ax2 = axes[0, 1]
ax2.plot(
    df_results["Batch Size"],
    df_results["Throughput (samples/s)"],
    marker="s",
    linestyle="-",
    linewidth=2,
    markersize=8,
    color="darkgreen"
)
ax2.set_xlabel("Batch Size", fontsize=11, fontweight="bold")
ax2.set_ylabel("Throughput (samples/s)", fontsize=11, fontweight="bold")
ax2.set_title("Inference Throughput vs Batch Size", fontsize=12, fontweight="bold")
ax2.grid(True, alpha=0.3)

# Plot 3: Bar chart - Latency comparison
ax3 = axes[1, 0]
bars = ax3.bar(
    [str(x) for x in df_results["Batch Size"]],
    df_results["Avg Latency (ms)"],
    color="coral",
    alpha=0.7,
    edgecolor="darkred",
    linewidth=1.5
)
ax3.set_xlabel("Batch Size", fontsize=11, fontweight="bold")
ax3.set_ylabel("Average Latency (ms)", fontsize=11, fontweight="bold")
ax3.set_title("Average Inference Latency by Batch Size", fontsize=12, fontweight="bold")
ax3.grid(True, alpha=0.3, axis="y")

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax3.text(
        bar.get_x() + bar.get_width()/2.,
        height,
        f"{height:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# Plot 4: Summary statistics table
ax4 = axes[1, 1]
ax4.axis("off")

# Create summary text
summary_text = f"""
BENCHMARK SUMMARY

Model: DistilBERT (Base)
Device: {'T4 GPU' if torch.cuda.is_available() else 'CPU'}
Sequence Length: 128 tokens
Number of Runs: 10 per batch size

BEST PERFORMANCE:
• Batch Size: {df_results.loc[df_results['Throughput (samples/s)'].idxmax(), 'Batch Size']:.0f}
• Throughput: {df_results['Throughput (samples/s)'].max():.2f} samples/s
• Latency: {df_results.loc[df_results['Throughput (samples/s)'].idxmax(), 'Avg Latency (ms)']:.2f} ms
• Tokens/sec: {df_results['Throughput (samples/s)'].max() * 128:.2f}

MEMORY EFFICIENCY:
• Model Size: {param_size:.2f} MB
• Peak GPU Memory: {peak_memory:.2f} MB (if GPU available)
"""

ax4.text(
    0.1, 0.9,
    summary_text,
    transform=ax4.transAxes,
    fontsize=10,
    verticalalignment="top",
    fontfamily="monospace",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5)
)

plt.tight_layout()
plt.show()

print("\n✓ Visualization complete!")
print("\nBenchmark finished successfully!")

## How to Use This Notebook in Google Colab

1. **Upload to Google Colab**: Open [Google Colab](https://colab.research.google.com) and upload this notebook
2. **Enable GPU**: Go to Menu → Runtime → Change runtime type → GPU (T4 preferred)
3. **Run All Cells**: Press Ctrl+F9 (or Cmd+F9 on Mac) or use Runtime → Run all
4. **Monitor Results**: Watch the benchmark progress and view the visualizations
5. **Customize**: Modify the `sample_texts` to test with your own text samples

### Expected Output
- GPU Information and memory details
- Benchmark results for different batch sizes (1, 4, 8, 16, 32)
- Performance metrics including latency, throughput, and efficiency
- 4 visualization plots showing performance trends

### Model Information
- **Model**: DistilBERT (Hugging Face)
- **Size**: ~268 MB (easily fits on T4 GPU with 16GB memory)
- **Speed**: 60% faster than BERT with 97% performance retention
- **Task**: Sequence Classification (can be modified for other tasks)

### Tips for Better Performance
- Use larger batch sizes (16-32) for better throughput
- The bottleneck is usually GPU memory, not computation
- DistilBERT is ideal for T4 GPUs; larger models may cause OOM errors
- For production use, consider quantization to reduce memory footprint